# Task 2B - 05 Policy trade-offs and what-ifs
This notebook turns the day into a decision: what exactly does each priority policy buy and cost, and what would releasing a workshop reefer change?
Every number comes from an exact solve.

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.append('../../tools/task2b')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, time
from task2b_common import *
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 50); pd.set_option('display.max_rows', 120)
plt.rcParams.update({'figure.dpi': 90, 'axes.grid': True, 'grid.alpha': .25})
for d in (METRICS, FIGURES, INTERIM): d.mkdir(parents=True, exist_ok=True)
scn, fleet, veh, dt, al = load_tables()
sub_team = pd.read_csv(SUBMISSION)            # the teammate's submitted allocation (read-only)
import task2b_exact as X
S = X.S


## 1. The policy frontier

In [2]:
pols = ['weighted', 'lexicographic', 'chilled_count', 'chilled_volume', 'max_orders']
res = {p: X.solve_exact(p, late_aware=False) for p in pols}      # fast reefer model: same served sets, no clock look-ahead
fr = pd.DataFrame({p: X.kpis(r) for p, r in res.items()}).T
fr.insert(0, 'rule', [X.POLICIES[p] for p in pols])
fr.to_csv(METRICS / 'exact_policy_frontier.csv'); fr[['rule', 'served', 'm3', 'chilled_served', 'chilled_m3', 'deferred_yday_served', 'priority_value']]

,rule,served,m3,chilled_served,chilled_m3,deferred_yday_served,priority_value
weighted,"weighted score (as in solve_task2b.py), solved...",79.0,328.30,21.0,140.72,9.0,1599.3
lexicographic,deferred-yesterday > chilled > days-since-serv...,77.0,313.83,19.0,126.25,10.0,1590.8
chilled_count,most chilled orders served first,79.0,328.30,21.0,140.72,9.0,1599.3
chilled_volume,most chilled m3 served first,77.0,331.35,19.0,143.77,9.0,1568.3
max_orders,most orders served first,79.0,328.30,21.0,140.72,9.0,1599.3


In [3]:
served_sets = {p: r.served_refs() for p, r in res.items()}
allrefs = set(S.ORDERS)
for p in pols: print('%-15s deferred: %s' % (p, sorted(allrefs - served_sets[p])))

weighted        deferred: ['S1-056', 'S1-058', 'S1-064', 'S1-067', 'S1-078', 'S1-083']
lexicographic   deferred: ['S1-003', 'S1-058', 'S1-064', 'S1-067', 'S1-071', 'S1-073', 'S1-075', 'S1-078']
chilled_count   deferred: ['S1-056', 'S1-058', 'S1-064', 'S1-067', 'S1-078', 'S1-083']
chilled_volume  deferred: ['S1-003', 'S1-005', 'S1-021', 'S1-056', 'S1-064', 'S1-067', 'S1-078', 'S1-083']
max_orders      deferred: ['S1-056', 'S1-058', 'S1-064', 'S1-067', 'S1-078', 'S1-083']


**Reading.** Five sensible rules give different answers only in the *chilled* column - everything ambient is served, apart from the physically impossible S1-078. The cheapest policy in orders is the weighted score / most-orders rule; the policy that never leaves a skipped outlet unserved twice costs two chilled orders and about 14 m3. There is no allocation that does both.

## 2. The deferred-yesterday trade-off, made exact
Outlet OUT074 (S1-083) was skipped yesterday and has now gone 5 days without a delivery. Serving it means a 173-minute outbound trip to Puttalam for a single 8.7 m3 stop, which uses 188 of one reefer's 270 minutes.

In [4]:
w, l = served_sets['weighted'], served_sets['lexicographic']
print('only in the weighted plan (served there, deferred under the lexicographic policy):', sorted(w - l))
print('only in the lexicographic plan:', sorted(l - w))
rows = [(r, S.ORDERS[r].outlet, S.ORDERS[r].district, round(S.ORDERS[r].m3, 1), S.ORDERS[r].deferred_yday, S.ORDERS[r].days_since) for r in sorted((w ^ l))]
pd.DataFrame(rows, columns=['order_ref', 'outlet', 'district', 'm3', 'deferred_yesterday', 'days_since'])

only in the weighted plan (served there, deferred under the lexicographic policy): ['S1-003', 'S1-071', 'S1-073', 'S1-075']
only in the lexicographic plan: ['S1-056', 'S1-083']


,order_ref,outlet,district,m3,deferred_yesterday,days_since
0,S1-003,OUT002,Colombo,1.8,0,1
1,S1-056,OUT053,Galle,3.8,0,2
2,S1-071,OUT065,Kurunegala,12.0,0,1
3,S1-073,OUT066,Kurunegala,5.8,0,1
4,S1-075,OUT067,Kurunegala,7.2,0,2
5,S1-083,OUT074,Puttalam,8.7,1,5


**Trade-off in one line.** To serve the outlet that has waited longest (and was skipped yesterday), the plan trades S1-003, S1-071, S1-073 and S1-075 for S1-083 and S1-056: four chilled orders wait instead of two. The four that wait are *first-time* misses (none was skipped yesterday). Nothing in the data says which is worse for the business; that is a **human decision**. The next section shows how close the weighted score already is to making the other choice.

## 3. Where does the weighted score flip?

In [5]:
def serves_083(w):
    r = X.solve_exact('weighted', weights=dict(deferred_yday=w), late_aware=False); return 'S1-083' in r.served_refs(), X.kpis(r)
rows = []
for wv in (0, 10, 20, 25, 30, 40, 60, 100):
    ok, k = serves_083(wv); rows.append((wv, ok, k['chilled_served'], k['served'], k['deferred_yday_served']))
flip = pd.DataFrame(rows, columns=['deferred_yday_weight', 'S1-083 served', 'chilled served', 'orders served', 'deferred-yesterday served']); flip

,deferred_yday_weight,S1-083 served,chilled served,orders served,deferred-yesterday served
0,0,False,21,79,9
1,10,False,21,79,9
2,20,False,21,79,9
3,25,False,21,79,9
4,30,True,19,77,10
5,40,True,19,77,10
6,60,True,19,77,10
7,100,True,19,77,10


In [6]:
lo, hi = 20.0, 30.0
for _ in range(7):
    mid = (lo + hi) / 2
    if serves_083(mid)[0]: hi = mid
    else: lo = mid
print('the weighted score serves S1-083 once the deferred-yesterday weight is about %.1f (first solver used 20, i.e. %.0f%% lower)' % (hi, 100 * (1 - 20 / hi)))
FLIP = hi
json.dump({'deferred_yday_flip_weight': round(hi, 2), 'first_solver_weight': 20}, open(METRICS / 'exact_flip.json', 'w'))

the weighted score serves S1-083 once the deferred-yesterday weight is about 27.8 (first solver used 20, i.e. 28% lower)


**Finding.** The first solver's weight of 20 sits below the point where the score would have served the five-day-unserved outlet (the threshold printed above, a little under 30). A weight roughly 40% larger gives the lexicographic answer. So the written priority order ("deferred yesterday first") and the numerical weights disagree by a small margin - which is exactly why the written policy should be implemented as an explicit ordering, not as tuned weights.

## 4. Value of releasing a workshop reefer

In [7]:
ws_reefers = [v for v in fleet[fleet.status == 'in_workshop'].vehicle_id if S.VEHICLES_ALL[v].temp == 'reefer']
rows = []
for pol in ('weighted', 'lexicographic'):
    base = X.kpis(res[pol]); base_served = res[pol].served_refs()
    rows.append((pol, 'none (today)', '', base['chilled_served'], base['chilled_m3'], base['served'], 0))
    for vid in ws_reefers:
        r = X.solve_exact(pol, extra_vehicles=(vid,), late_aware=False); k = X.kpis(r)
        rows.append((pol, vid, '%s %.1f m3' % (S.VEHICLES_ALL[vid].type, S.VEHICLES_ALL[vid].m3_cap), k['chilled_served'], k['chilled_m3'], k['served'], k['served'] - base['served']))
wi = pd.DataFrame(rows, columns=['policy', 'released reefer', 'class', 'chilled orders served', 'chilled m3 served', 'orders served', 'net orders gained'])
wi.to_csv(METRICS / 'exact_whatif_workshop_reefer.csv', index=False); wi

,policy,released reefer,class,chilled orders served,chilled m3 served,orders served,net orders gained
0,weighted,none (today),,21,140.72,79,0
1,weighted,VEH001,truck 26.4 m3,23,167.24,81,2
2,weighted,VEH002,truck 21.1 m3,23,163.88,81,2
3,weighted,VEH004,truck 33.4 m3,23,167.81,81,2
4,weighted,VEH005,truck 33.4 m3,23,167.81,81,2
5,weighted,VEH035,van 7.0 m3,23,149.67,81,2
6,lexicographic,none (today),,19,126.25,77,0
7,lexicographic,VEH001,truck 26.4 m3,23,167.24,81,4
8,lexicographic,VEH002,truck 21.1 m3,23,163.88,81,4
9,lexicographic,VEH004,truck 33.4 m3,23,167.81,81,4


**Finding.** Releasing **any one of the four workshop reefer trucks** (VEH001, VEH002, VEH004, VEH005) lifts both policies to the same result - 23 of 26 chilled orders and 81 orders served - so with one more truck the weighted score and the "never miss twice" policy stop disagreeing at all. The workshop reefer van (VEH035) helps less. This is the cheapest lever for tomorrow's peak and is quantified here exactly, not guessed.

## What needs a human decision
1. **Policy:** weighted score (79 orders, one outlet missed twice) or lexicographic "never miss an outlet twice" (77 orders, none missed twice). Section 3 shows the margin between them is small.
2. **Count or volume** for the "chilled" level: we rank by number of chilled orders (outlets kept in stock); ranking by chilled m3 (`chilled_volume` above) gives a different set.
3. **Which workshop reefer to repair first**, and whether a hired vehicle is an option (section 4 gives the value of each).